# From a PDF to a table: a classical OCR, an LLM, and a loop

Session 4, Part 4. Five historical PDFs of different kinds, from the Internet Archive:

| | Document | Why it is here |
| --- | --- | --- |
| 1760 | A handwritten report to the Académie des sciences (BIU Santé, Paris) | handwriting |
| 1793 | A printed decree of the département des Vosges (Newberry Library) | old type, the long s (ſ) |
| 1958 | A candidate's leaflet, legislative election (Archelec, Sciences Po) | letterpress, columns |
| 1978 | A candidate's leaflet (Archelec) | offset, photos, slogans |
| 1993 | A candidate's leaflet (Archelec) | modern layout |

The pipeline, in five steps:

1. **Look** at the PDF, here in the notebook.
2. **A classical OCR** (Tesseract): a table of words, each with a position and a confidence.
3. **An LLM** reads the same page: a transcription, and a small form (type, date, title, people, places).
4. **Compare** the two readings.
5. **A loop**: every page of every document, into one table. That is how it scales.

**Before you start.** Colab Secrets (key icon, left sidebar): `ANTHROPIC_API_KEY`, "Notebook access" on. Never paste a key into a cell. A CPU runtime is enough.

The Archelec leaflets are under a restrictive licence (CC BY-NC-ND): look at them, do not republish them.

## 0. Setup (run once; nothing to explain here)

In [ ]:
!apt-get -qq install -y tesseract-ocr tesseract-ocr-fra > /dev/null
%pip install -q pymupdf pytesseract anthropic pydantic

In [ ]:
import base64, datetime, difflib, hashlib, io, json, os, pathlib, re, time, unicodedata
from concurrent.futures import ThreadPoolExecutor

import pandas as pd
import pymupdf                      # opens PDFs and turns pages into images
import pytesseract                  # the Python door to Tesseract, the classical OCR
import requests
from IPython.display import display
from PIL import Image, ImageDraw
from pydantic import BaseModel, ConfigDict

# ---- Parameters: everything a referee would ask about is here ----
DOCS = [
    {"id": "manuscript_1760", "kind": "handwriting",
     "url": "https://archive.org/download/BIUSante_ms05541x01/BIUSante_ms05541x01.pdf"},
    {"id": "decree_1793", "kind": "old print, long s",
     "url": "https://archive.org/download/extraitduregistr00vosg_0/extraitduregistr00vosg_0.pdf"},
    {"id": "leaflet_1958", "kind": "letterpress leaflet",
     "url": "https://archive.org/download/EL010_L_1958_11_062_14_1_PF_03/EL010_L_1958_11_062_14_1_PF_03.pdf"},
    {"id": "leaflet_1978", "kind": "offset leaflet",
     "url": "https://archive.org/download/EL105_L_1978_03_073_03_1_PF_03/EL105_L_1978_03_073_03_1_PF_03.pdf"},
    {"id": "leaflet_1993", "kind": "modern leaflet",
     "url": "https://archive.org/download/EL191_L_1993_03_042_06_1_PF_01/EL191_L_1993_03_042_06_1_PF_01.pdf"},
]
MAX_PAGES = 3                  # pages read per document (the first ones)
DPI = 200                      # resolution of the page images given to Tesseract
LANG = "fra"                   # Tesseract's language model
LOW_CONF = 60                  # below this confidence (0 to 100), a word is flagged
CLAUDE_MODEL = "claude-haiku-5-5"
LLM_SIDE = 1568                # longest side, in pixels, of the image sent to the model
PRICE = {"in": 0.10, "out": 0.50}    # $ per million tokens, November 2026

WORK = pathlib.Path("s4_ocr_work")
(WORK / "pdf").mkdir(parents=True, exist_ok=True)
BY_ID = {d["id"]: d for d in DOCS}


def get_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)


def fetch_pdf(doc_id):
    """The PDF, as bytes. Downloaded once, then read from disk."""
    f = WORK / "pdf" / f"{doc_id}.pdf"
    if not f.exists():
        for i in range(4):
            r = requests.get(BY_ID[doc_id]["url"], timeout=120, headers={"User-Agent": "uzh-llm-socsci-2026 (course notebook)"})
            if r.status_code == 200:
                f.write_bytes(r.content)
                break
            time.sleep(2 ** i)
        else:
            raise RuntimeError(f"could not download {doc_id}: HTTP {r.status_code}")
    return f.read_bytes()


def page_images(doc_id, dpi=DPI):
    """The first MAX_PAGES pages of a PDF, as images."""
    pdf = pymupdf.open(stream=fetch_pdf(doc_id), filetype="pdf")
    out = []
    for page in list(pdf)[:MAX_PAGES]:
        pix = page.get_pixmap(dpi=dpi, colorspace=pymupdf.csRGB, alpha=False)
        out.append(Image.frombytes("RGB", (pix.width, pix.height), pix.samples))
    return out


def show(img, width=520):
    display(img.resize((width, int(img.height * width / img.width))))


print("ready")

## 1. Look at the PDF

A PDF is a container. Inside: either **text** (a file written on a computer), or **pictures of pages** (a scan). Only the first can be read directly.

In [ ]:
DOC = "leaflet_1958"
pdf = pymupdf.open(stream=fetch_pdf(DOC), filetype="pdf")
len(pdf), [len(page.get_images()) for page in pdf]      # number of pages; pictures on each page

Is there a text layer? Some scans carry the text of an earlier OCR, hidden behind the picture.

In [ ]:
pdf[0].get_text()[:300]

The first page, as a picture. This is what Tesseract and the model will see.

In [ ]:
PAGES = page_images(DOC)
PAGE = PAGES[0]
show(PAGE)

Change `DOC` above to `manuscript_1760` or `decree_1793`, rerun the three cells, and look. Then come back to `leaflet_1958`.

## 2. A classical OCR: Tesseract

Tesseract finds blocks, lines, then words, and recognises each word from its shape. It runs here, on this machine: free, and the same answer every time. What it returns is **a table**: one row per word.

In [ ]:
def tesseract_words(img):
    """One row per word: the text, where it is on the page, and Tesseract's confidence (0 to 100)."""
    t = pytesseract.image_to_data(img, lang=LANG, output_type=pytesseract.Output.DATAFRAME)
    t = t[(t.conf >= 0) & t.text.notna()].copy()
    t["text"] = t.text.astype(str).str.strip()
    return t[t.text != ""][["block_num", "par_num", "line_num", "word_num", "left", "top", "width", "height", "conf", "text"]].reset_index(drop=True)


WORDS = tesseract_words(PAGE)
len(WORDS)

In [ ]:
WORDS.head(12)

Its own doubt: the confidence. The mean over the page, and the words it is least sure of.

In [ ]:
round(WORDS.conf.mean(), 1), round((WORDS.conf < LOW_CONF).mean(), 2)     # mean confidence; share of flagged words

In [ ]:
WORDS.sort_values("conf").head(15)[["conf", "text"]].T

Where are they on the page? In red, the words under `LOW_CONF`.

In [ ]:
def flag(img, words, threshold=LOW_CONF):
    out = img.copy()
    draw = ImageDraw.Draw(out)
    for w in words[words.conf < threshold].itertuples():
        draw.rectangle([w.left, w.top, w.left + w.width, w.top + w.height], outline=(200, 30, 30), width=4)
    return out


show(flag(PAGE, WORDS))

From the table of words back to a text: the words of a line, in order; the lines of a block, in order.

In [ ]:
def text_of(words):
    lines = words.groupby(["block_num", "par_num", "line_num"], sort=True).text.apply(" ".join)
    return "\n".join(lines)


TESS_TEXT = text_of(WORDS)
print(TESS_TEXT[:1200])

**From a text to variables, with code.** A year is easy: four digits. A name is not: "a word with a capital letter" catches the candidate, the town, and the first word of every sentence.

In [ ]:
re.findall(r"\b1[5-9]\d\d\b", TESS_TEXT), re.findall(r"\b[A-ZÉÈ][A-ZÉÈ\-]{3,}\b", TESS_TEXT)[:15]

## 3. An LLM reads the same page

The picture of the page goes to the model with instructions, and a form to fill in. The form has two parts: the **transcription**, and a few **variables** that code alone could not find.

In [ ]:
import anthropic

assert get_secret("ANTHROPIC_API_KEY"), "No key: add ANTHROPIC_API_KEY in Colab Secrets (key icon), Notebook access on"
claude = anthropic.Anthropic(api_key=get_secret("ANTHROPIC_API_KEY"), max_retries=5)


class PageForm(BaseModel):
    model_config = ConfigDict(extra="forbid")
    transcription: str
    document_type: str
    date: str
    title: str
    people: list[str]
    places: list[str]


INSTRUCTIONS = """You read one page of a scanned historical French document and fill in the form.

transcription: the text of the page, word for word, with its spelling, capitals and accents. Do not correct, do not modernise, do not translate, do not summarise. Write the long s as s. Never complete a word or a sentence you cannot read: write [illisible] in its place. Several columns: one after the other, in reading order. Keep the line breaks of the original, and one blank line between blocks of text. Ignore library stamps and colour charts. An empty page: an empty string.

Then, from this page only, and never from your own knowledge:
document_type: a few words, for example "election leaflet", "administrative decree", "handwritten report". "" if you cannot tell.
date: the date printed or written on the page, as it appears. "" if none.
title: the title or heading of the document, as printed. "" if none.
people: the names of the persons mentioned, as printed. An empty list if none.
places: the towns, départements, and countries mentioned, as printed. An empty list if none."""


def to_jpeg(img, side=LLM_SIDE):
    img = img.copy()
    img.thumbnail((side, side))
    buf = io.BytesIO()
    img.save(buf, "JPEG", quality=85)
    return base64.standard_b64encode(buf.getvalue()).decode()


def claude_page(img):
    r = claude.messages.parse(
        model=CLAUDE_MODEL, max_tokens=8000, system=INSTRUCTIONS, thinking={"type": "disabled"},
        messages=[{"role": "user", "content": [
            {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": to_jpeg(img)}},
            {"type": "text", "text": "Fill in the form for this page."}]}],
        output_format=PageForm)
    form = r.parsed_output.model_dump() if r.parsed_output else None
    return {"form": form, "stop": r.stop_reason, "in": r.usage.input_tokens, "out": r.usage.output_tokens}

One page. `stop` should be `end_turn`: `max_tokens` would mean the answer was cut.

In [ ]:
ONE = claude_page(PAGE)
ONE["stop"], ONE["in"], ONE["out"]

In [ ]:
LLM_TEXT = ONE["form"]["transcription"]
print(LLM_TEXT[:1200])

And the variables: one row of a table, from one page.

In [ ]:
pd.DataFrame([{k: v for k, v in ONE["form"].items() if k != "transcription"}]).T

## 4. Compare the two readings

Nobody typed this page by hand: there is no ground truth. What we can do: measure how much the two readings differ, look at where, and go back to the picture.

In [ ]:
def norm(s):
    """Lower case, no accents, no punctuation: only the words are compared."""
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9 ]", " ", s)).strip()


def agreement(a, b):
    """Share of words the two texts have in common, in the same order (0 to 1)."""
    a, b = norm(a).split(), norm(b).split()
    return difflib.SequenceMatcher(None, a, b, autojunk=False).ratio() if a or b else 1.0


round(agreement(TESS_TEXT, LLM_TEXT), 2)

The words only one of them has. Read both lists with the page in front of you: who is right?

In [ ]:
def only(a, b, k=30):
    return sorted(set(norm(a).split()) - set(norm(b).split()), key=len, reverse=True)[:k]


print("only Tesseract:", only(TESS_TEXT, LLM_TEXT))
print("only the model:", only(LLM_TEXT, TESS_TEXT))

Line by line, where they differ (`-` Tesseract, `+` the model).

In [ ]:
def diff(a, b, k=25):
    a = [l.strip() for l in a.splitlines() if l.strip()]
    b = [l.strip() for l in b.splitlines() if l.strip()]
    return [l for l in difflib.unified_diff(a, b, lineterm="", n=0) if l[:1] in "+-" and l[:3] not in ("+++", "---")][:k]


diff(TESS_TEXT, LLM_TEXT)

**What to look for.** Tesseract's errors look like errors: broken words, stray signs. The model's errors look like French: a corrected spelling, a completed line, a word that is not on the page. The second kind is harder to see, and worse.

## 5. The loop: every page, one table

One function does everything for one page. The loop calls it for every page of every document. Each answer is written to a file as it arrives: a rerun does not pay twice, and a crash loses nothing. On 5 documents or 5,000, the code is the same.

In [ ]:
CONFIG = hashlib.sha256((INSTRUCTIONS + CLAUDE_MODEL + str(DPI) + LANG).encode()).hexdigest()[:8]
RAW = WORK / f"pages_{CONFIG}.jsonl"          # every answer, kept


def process(task):
    doc_id, page_no, img = task
    t0 = time.time()
    words = tesseract_words(img)
    t1 = time.time()
    try:
        llm = claude_page(img)
    except Exception as e:
        llm = {"form": None, "error": repr(e)[:300]}
    return {"doc_id": doc_id, "page": page_no, "tess_text": text_of(words), "tess_words": len(words),
            "tess_conf": round(float(words.conf.mean()), 1) if len(words) else None,
            "tess_seconds": round(t1 - t0, 1), "llm_seconds": round(time.time() - t1, 1), **llm}


DONE = {}
if RAW.exists():
    for line in RAW.read_text().splitlines():
        rec = json.loads(line)
        if rec.get("form"):
            DONE[(rec["doc_id"], rec["page"])] = rec

TASKS = [(d["id"], n, img) for d in DOCS for n, img in enumerate(page_images(d["id"])) if (d["id"], n) not in DONE]
print(len(DONE), "pages already done,", len(TASKS), "to do")
with ThreadPoolExecutor(4) as pool, RAW.open("a") as f:
    for rec in pool.map(process, TASKS):
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        DONE[(rec["doc_id"], rec["page"])] = rec
        print(" ", rec["doc_id"], rec["page"], "ok" if rec.get("form") else rec.get("error"))

The table: one row per page. The classical OCR on the left, the model on the right, their agreement in the middle.

In [ ]:
def row(rec):
    f = rec.get("form") or {}
    return {"doc_id": rec["doc_id"], "page": rec["page"], "kind": BY_ID[rec["doc_id"]]["kind"],
            "tess_words": rec["tess_words"], "tess_conf": rec["tess_conf"],
            "agreement": round(agreement(rec["tess_text"], f.get("transcription", "")), 2),
            "llm_words": len(f.get("transcription", "").split()), "illisible": f.get("transcription", "").count("[illisible]"),
            "type": f.get("document_type"), "date": f.get("date"), "title": (f.get("title") or "")[:60],
            "people": "; ".join(f.get("people", []))[:80], "places": "; ".join(f.get("places", []))[:60]}


order = {d["id"]: k for k, d in enumerate(DOCS)}
TABLE = pd.DataFrame([row(r) for r in sorted(DONE.values(), key=lambda r: (order.get(r["doc_id"], 99), r["page"]))])
pd.set_option("display.max_colwidth", 80)
TABLE

**Read the table by kind of document.** Where is Tesseract confident, and where does it give up? Where do the two readings agree? On the handwriting, the classical OCR returns almost nothing: the comparison then says nothing about the model, and only your eyes can check it.

In [ ]:
TABLE.groupby("kind", sort=False)[["tess_conf", "agreement", "tess_words", "llm_words"]].mean().round(2)

One page where they disagree most: the picture, then both texts.

In [ ]:
worst = TABLE.sort_values("agreement").iloc[0]
rec = DONE[(worst.doc_id, int(worst.page))]
print(worst.doc_id, "page", worst.page, "| agreement", worst.agreement, "| Tesseract confidence", worst.tess_conf)
show(page_images(worst.doc_id)[int(worst.page)])

In [ ]:
print("TESSERACT\n", rec["tess_text"][:700], "\n\nTHE MODEL\n", (rec["form"] or {}).get("transcription", "")[:700])

### What it costs, and how it scales

Time and money per page, from this run. Then the same loop on a real corpus: the 4,010 leaflets of the 1978 election have about two pages each.

In [ ]:
recs = [r for r in DONE.values() if r.get("form")]
cost = sum(r["in"] * PRICE["in"] + r["out"] * PRICE["out"] for r in recs) / 1e6
per_page = {"tesseract seconds": round(sum(r["tess_seconds"] for r in recs) / len(recs), 1),
            "model seconds": round(sum(r["llm_seconds"] for r in recs) / len(recs), 1),
            "model dollars": round(cost / len(recs), 4)}
print(per_page)
print(f"8,000 pages: Tesseract about {per_page['tesseract seconds'] * 8000 / 3600:.1f} hours on one core, free; "
      f"the model about ${per_page['model dollars'] * 8000:.0f}")

**At scale, three things change, not the code.**

- **Tesseract**: one core per page. On the ScienceCluster, a job with 32 cores reads 32 pages at a time (`sessions/04/cluster/`).
- **The model**: the batch API takes all the requests at once, at half price, and answers within a day (Session 3).
- **You**: you cannot read 8,000 pages. You read a random sample, and you let the two readings point at the pages where they disagree.

Keep what a referee would ask for: the table, every raw answer, the instructions, the model, the date.

In [ ]:
TABLE.to_csv(WORK / "table.csv", index=False)
(WORK / "params.json").write_text(json.dumps({
    "date_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(), "documents": DOCS, "max_pages": MAX_PAGES,
    "tesseract": {"version": str(pytesseract.get_tesseract_version()), "lang": LANG, "dpi": DPI},
    "llm": {"model": CLAUDE_MODEL, "image_side": LLM_SIDE, "instructions_sha256": hashlib.sha256(INSTRUCTIONS.encode()).hexdigest()},
    "raw_answers": RAW.name}, indent=2, ensure_ascii=False))
sorted(p.name for p in WORK.iterdir())

## 6. Your own PDF

Add a line to `DOCS` with the address of a PDF, or upload one (Colab: Files panel) and give its path. Then rerun Part 5: the loop only reads what is new.

In [ ]:
MY_PDF = ""          # a URL, or the path of a file you uploaded, e.g. "my_scan.pdf"
if MY_PDF:
    doc = {"id": pathlib.Path(MY_PDF).stem, "kind": "mine", "url": MY_PDF}
    if not MY_PDF.startswith("http"):
        (WORK / "pdf" / f"{doc['id']}.pdf").write_bytes(pathlib.Path(MY_PDF).read_bytes())
    if doc["id"] not in BY_ID:
        DOCS.append(doc)
        BY_ID[doc["id"]] = doc
    show(page_images(doc["id"])[0])

**What you just did.** A PDF opened and looked at; a classical OCR that returns a table of words with its own doubts; a model that returns a transcription and variables; two readings compared without a ground truth; and a loop that turns pages into rows. In Part 5 of the session, the same idea on 300 leaflets: a form, filled in by two models, checked against a catalogue.